# Hypothesis Testing and Predictive Modelling ( Global CO2 Responsibility)

## Objective
-Statistically test key project hypotheses (trend + relationships)
- Build simple predictive models to support forward-looking insights

## Inputs
-Clean dataset: df_co2_emissions_clean (1990-2024)

## Outputs
- Test results (p-values, correlations)
- Model results (MAE/RMSE/R²)
- Export CSVs for Power BI

In [3]:
import pandas as pd
import numpy as np      
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats 
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression   
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline   

In [4]:
# Load the dataset
df_co2_emissions = pd.read_csv('../DataSet/Cleaned/CO2_Emissions_Cleaned.csv')
# Displaying the first few rows of the dataset
df_co2_emissions.head()



,country,iso_code,year,co2,co2_per_capita,gdp,population,cumulative_co2
0,Afghanistan,AFG,1990,2.024326,0.168054,1.306598e+10,12045664.0,58.603493
1,Afghanistan,AFG,1991,1.914301,0.156411,1.204736e+10,12238879.0,60.517792
2,Afghanistan,AFG,1992,1.482054,0.111609,1.267754e+10,13278983.0,61.999847
3,Afghanistan,AFG,1993,1.486943,0.099506,9.834582e+09,14943175.0,63.486794
4,Afghanistan,AFG,1994,1.453829,0.089462,7.919856e+09,16250800.0,64.940620


In [5]:
# Checking datset information
df_co2_emissions.info() 
df_co2_emissions.describe() 




<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7508 entries, 0 to 7507
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   country         7508 non-null   object 
 1   iso_code        7508 non-null   object 
 2   year            7508 non-null   int64  
 3   co2             7508 non-null   float64
 4   co2_per_capita  7442 non-null   float64
 5   gdp             5409 non-null   float64
 6   population      7442 non-null   float64
 7   cumulative_co2  7508 non-null   float64
dtypes: float64(5), int64(1), object(2)
memory usage: 469.4+ KB


,year,co2,co2_per_capita,gdp,population,cumulative_co2
count,7508.000000,7508.000000,7442.000000,5.409000e+03,7.442000e+03,7508.000000
mean,2007.035828,137.544464,5.022831,4.895098e+11,3.179892e+07,5822.072674
std,10.083330,675.286457,7.795041,1.705000e+12,1.262322e+08,27988.146188
min,1990.000000,0.000000,0.000000,2.571720e+08,1.776000e+03,0.000000
25%,1998.000000,0.852971,0.714055,1.886576e+10,7.567225e+05,23.068187
50%,2007.000000,7.172069,2.744757,6.111433e+10,5.713997e+06,227.102509
75%,2016.000000,53.937630,6.832723,2.910000e+11,2.045923e+07,1978.472718
max,2024.000000,12289.037110,364.790833,2.700000e+13,1.450936e+09,434866.562500


In [6]:
# Identify latest year in the dataset
latest_year = df_co2_emissions['year'].max()

#Latest year dataset
df_latest = df_co2_emissions[df_co2_emissions['year'] == latest_year].copy()

# Global yearly totals
global_year = (df_co2_emissions.groupby('year', as_index=False)['co2'].sum())

latest_year, df_latest.shape,  global_year.shape


(2024, (215, 8), (35, 2))

## Hypothesis Testing
Hypothesis 1 - Has Global CO2 increased since 1990?

Ho: Global CO2 emissions have not increased over time
H1: Global CO2 emissions have increased over time


Pearson correlation analysis was used to examine the relationship between time (year) and global CO₂ emissions. This statistical test is appropriate because both variables are continuous and normally distributed at the aggregate level. Pearson’s r measures the strength and direction of a linear relationship, producing a coefficient (r) ranging from -1 to +1

In [7]:
# Testing using Pearson Correlation
r_h1, p_h1 = stats.pearsonr(global_year['year'], global_year['co2'])
print('H1 pearson r:', round(r_h1, 3))
print(' H1 p-value:', p_h1)

H1 pearson r: 0.982
 H1 p-value: 1.8306757151155014e-25


 ## H1 Result and Interpretation 
Pearson correlation analysis revealed a strong positive relationship between year and global CO2 emissions (r = 0.982, p < 0.05).
This indicates that as time progresses, global emissions increase significantly. The strength of the correlation suggests that the rise in emissions is not random but follows a consistent upward trajectory.
From an analytical perspective, this trend reflects the long-term expansion of industrial activity, fossil fuel dependence, and economic growth worldwide. Although short-term declines are visible, such as the temporary drop around 2020,  the overall pattern demonstrates sustained growth in atmospheric carbon output.

## Conclusion
The null hypothesis is therefore rejected. There is sufficient statistical evidence to conclude that global CO₂ emissions have increased significantly since 1990. This finding reinforces the urgency of climate mitigation policies, as historical trends show continued escalation rather than stabilisation.

## Hypothesis 2
H2:Countries with larger populations produce higher total CO2

Ho (Null Hypothesis): There is no statistically significant relationship between a country's GDP and its total CO2 emissions
H1(Alternative Hypothesis) : There is a statistically significant relationship between GDP and total CO2 emissions


Pearson correlation was selected because:
Both variables (GDP and CO₂ emissions) are continuous numerical variables.
The objective is to measure the strength and direction of the linear relationship.
The dataset is sufficiently large (213 countries), making Pearson appropriate and reliable.


In [8]:
# Clean subset for H2
df_h2 = df_latest.dropna(subset=['population', 'co2']).copy()
# Testing using Pearson Correlation
r_h2, p_h2 = stats.pearsonr(df_h2['population'], df_h2['co2'])
print('H2 pearson r:', round(r_h2, 3))
print('H2 p-value:', p_h2)  
print ('Countries used', df_h2['country'].nunique())

H2 pearson r: 0.824
H2 p-value: 5.786672971691484e-54
Countries used 213


## Hypothesis 2 Interpretation
The Pearson correlation coefficient of 0.824 indicates a strong positive relationship between GDP and total CO₂ emissions.
This means that:
Countries with larger economies tend to produce higher total emissions.
Economic output and industrial activity are closely linked to carbon production.
As GDP increases, emissions generally increase as well.
The p-value is far below the standard significance level of 0.05, meaning the result is statistically significant.

Therefore, the null hypothesis is rejected.



# Hypothesis 2 Conclusion 
The hypothesis test confirms a strong and statistically significant relationship between GDP and CO₂ emissions.

While population explains emission scale, GDP reflects the economic intensity behind carbon output. Together, both factors provide a deeper understanding of global emissions responsibility.

## Hypothesis 3
H3: Developed vs Developing : Per-Capita CO2 Emissions

Ho ( Null Hypothesis) : There is no statistically significant difference in average CO2 emissions per capita between developed and developing countries.

H1 ( Alternative Hypothesis): There is a statistically significant difference in average CO2 emissions per capita between developed and developing countries.

Statistical Test Used

An independent samples t-test (Welch’s t-test) was conducted to compare mean per-capita CO2 emissions between developed and developing countries.
This test was selected because:

Two independent groups were being compared.

The sample sizes were unequal.

Variance between groups could differ.

Welch’s t-test is more robust under these conditions.

In [9]:
# Creating a mapping of countries to their development status
development_status_mapping = {
    'Qatar': 'Developed',
    'Kuwait': 'Developed',  
    'Brunei': 'Developed',
    'Bahrain': 'Developed',
    'United Arab Emirates': 'Developed',
    'New Caledonia': 'Developed',
    'Saudi Arabia': 'Developing',
    'Oman': 'Developing',
    'Trinidad and Tobago': 'Developing',
    'Sint Maarten (Dutch part)': 'Developing'}




In [10]:
# Adding development status to the dataset
h3_df = df_latest[['country', 'co2_per_capita']].copy()
# Add development status to the dataset
h3_df['Development_Status'] = h3_df['country'].map(development_status_mapping)
# Kepp only the 10 mapped countries and  remove missing per-capita values
h3_df = h3_df.dropna(subset=['Development_Status', 'co2_per_capita'])
h3_df


,country,co2_per_capita,Development_Status
559,Bahrain,24.270082,Developed
1084,Brunei,26.046202,Developed
3627,Kuwait,26.247530,Developed
4812,New Caledonia,18.064400,Developed
5127,Oman,15.651107,Developing
5512,Qatar,41.271179,Developed
5897,Saudi Arabia,20.379194,Developing
6107,Sint Maarten (Dutch part),16.546274,Developing
6842,Trinidad and Tobago,22.931944,Developing
7122,United Arab Emirates,20.131075,Developed


In [11]:
# Spliting the groups
developed = h3_df[h3_df['Development_Status'] == 'Developed']['co2_per_capita']
developing = h3_df[h3_df['Development_Status'] == 'Developing']['co2_per_capita']
print('Developed count:', len(developed))
print('Developing count:', len(developing))

Developed count: 6
Developing count: 4


In [12]:
# Run t-test
t_stat, p_value = stats.ttest_ind(developed, developing, equal_var=False)
print('T-statistic:', t_stat)
print('P-value:', p_value)

T-statistic: 1.904962196125825
P-value: 0.09773905692158716


## Hypothesis 3 Interpretation
The independent samples t-test produced a p-value of 0.0977, which is above the standard 0.05 significance level. Therefore, the null hypothesis cannot be rejected.

Although developed countries show a higher average per-capita CO2 emission compared to developing countries, this difference is not statistically significant within the selected sample.

This suggests that, among the top per-capita emitters analysed, development status alone does not fully explain variation in emissions intensity.


## Hypothesis 3 Conclusion
The lack of statistical significance may be influenced by the limited sample size and the fact that the dataset focuses only on countries with exceptionally high per-capita emissions. A broader global comparison may yield stronger statistical separation between development groups.

# Predictive Modelling

## Objective: Build a simple regression model to forcast global CO2 emissions beyond 2024

We would prepare global yearly totals, fit linear regressin, evalust model, forecast 2025- 2030. 


In [13]:
# Prepare Global Data
global_year.head()


,year,co2
0,1990,22184.273978
1,1991,22662.921003
2,1992,21966.606838
3,1993,22192.437121
4,1994,22389.768557


In [14]:
# Define features and target variable
X = global_year[['year']]   
y = global_year['co2']

Model Training Approach

A train–test split was not applied in this time-series regression model. Unlike cross-sectional predictive modelling, time-series forecasting relies on the full chronological sequence to estimate trend behaviour accurately. Given the relatively small sample size (35 yearly observations), retaining the full dataset ensured maximum information for trend estimation.

In [15]:
# Train Linear Regression Model
model_global = LinearRegression()
model_global.fit(X, y)

LinearRegression()

In [16]:
# slope and intercept
slope = model_global.coef_[0]
intercept = model_global.intercept_

print('Slope (annual change in CO2 emissions):', slope)
print('Intercept:', intercept)

Slope (annual change in CO2 emissions): 515.19516090648
Intercept: -1004491.4355783222


## Forecast Model Interpretation
A linear regression model was developed to forecast global CO2 emissions based on historical trends from 1990 to 2024.

The model estimated an annual increase of approximately 515 million tonnes of CO2 per year, highlighting a sustained upward emissions trajectory over the last three decades.

This growth reflects continued industrial expansion, fossil fuel reliance, and global economic development. While short-term fluctuations exist, the long-term direction remains consistently upward.

In [17]:
# Prediction using Mean Square Error
predictions = model_global.predict(X)
mse = mean_squared_error(y, predictions)
r2 = r2_score(y, predictions)
print('Mean Squared Error:', mse)
print('R-squared:', r2)

Mean Squared Error: 1001834.8786141803
R-squared: 0.9643161372829833


R-Squared interpretation
An R-Squared value of 0.964 indicates that 96.4% of the variation in global CO2 emissions is explained by time (year) in the model.

This reflects an extremely strong linear relationship between time and emissions growth.

Emissions have followed a highly predictable upward trajectory over the past three decades.

In [18]:
# Forecasting future emissions from 2025 to 2031  
future_years = pd.DataFrame({'year': list(range(2025, 2031))})        
future_years['predicted_co2'] = model_global.predict(future_years)
future_years


,year,predicted_co2
0,2025,38778.765257
1,2026,39293.960418
2,2027,39809.155579
3,2028,40324.350740
4,2029,40839.545901
5,2030,41354.741062


## Predictive Model Interpretation

The regression model projects that global CO2 emissions will continue increasing if historical patterns persist.
This indicates that global emissions could exceed 41,000 million tonnes by 2030 under a business-as-usual trajectory.

This sustained increase reflects:
Continued fossil fuel dependence,
Expanding industrial production,
Population and economic growth,
Uneven adoption of renewable energy.
The projection reinforces the urgency of global decarbonisation strategies.


## Model Limitation
Linear regression assumes emissions will continue rising at a constant rate. In reality, emissions may stabilise or decline due to climate agreements, renewable energy transitions, or economic restructuring. Therefore, this forecast represents a continuation of historical trends rather than a policy-adjusted prediction.

## Predictive Modelling Conclusion
The forecast strengthens the dashboard’s responsibility narrative by showing that, without structural intervention, cumulative emissions, and therefore historical accountability will continue expanding.

## Linear Regression Model 
## Model B1

A simple linear regression model was developed to examine the predictive relationship between national population size and total CO2 emissions.

The objective was to determine whether population alone can explain emissions responsibility or whether additional socio-economic factors contribute to emissions variation.

Linear regression was selected due to its suitability for analysing relationships between continuous variables. Population and CO₂ emissions are both quantitative measures, making regression modelling appropriate for estimating predictive strength and direction.



In [19]:
# Prepare latest year data for modelling
df_reg = df_co2_emissions[df_co2_emissions['year'] == 2024].dropna(subset=['population', 'co2'])

In [20]:
# Define features and target variable
X = df_reg[['population']]
y = df_reg['co2']   

In [21]:
# Split Train and Test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [22]:
# Train Linear Regression Model
model_pop = LinearRegression()
model_pop.fit(X_train, y_train)


LinearRegression()

In [23]:
# Generate predictions 
Pred = model_pop.predict(X_test)



In [24]:
# Evaluate Model
mae_pop = mean_absolute_error(y_test, Pred)
rmse_pop = np.sqrt(mean_squared_error(y_test, Pred))
r2_pop = r2_score(y_test, Pred)
print('MAE', mae_pop)
print('RMSE', rmse_pop)
print('R-squared', r2_pop)

MAE 89.6757982240186
RMSE 166.90780513013524
R-squared -0.42939466180362174


## Model B Interpretation
A simple linear regression model was developed to predict national CO₂ emissions based on population size.

Although earlier correlation analysis indicated a strong positive association (r = 0.824), the predictive model achieved an R² value of -0.429 when evaluated on unseen data. This indicates that population alone is insufficient for accurately predicting emissions levels.

The MAE and RMSE values further suggest substantial prediction errors across countries.

This result demonstrates that emissions responsibility cannot be explained solely by population size. Industrial structure, energy mix, economic output, and technological development likely play significant roles in determining national emissions levels.

A negative R² indicates that the model performs worse than predicting the mean emissions value, highlighting limited predictive power from population alone.



## Multi-factor Regression
## Model B2

In [32]:
#loading the dataset for multi regression
df_co2_emissions = pd.read_csv('../DataSet/Cleaned/CO2_Emissions_Cleaned.csv')

In [69]:
df_co2_emissions.tail()

,country,iso_code,year,co2,co2_per_capita,gdp,population,cumulative_co2
7503,Zimbabwe,ZWE,2020,8.490839,0.546847,2.317871e+10,15526887.0,783.314758
7504,Zimbabwe,ZWE,2021,10.222778,0.647125,2.514009e+10,15797220.0,793.537537
7505,Zimbabwe,ZWE,2022,12.231845,0.761205,2.590159e+10,16069061.0,805.769409
7506,Zimbabwe,ZWE,2023,13.443295,0.822681,NaN,16340829.0,819.212647
7507,Zimbabwe,ZWE,2024,13.701154,0.823666,NaN,16634366.0,832.913879


In [82]:
# Prepare data for modelling
# Get latest year per country
# Filter for rows with GDP, then take the last year per country
df_latest_gdp = (
    df_co2_emissions.dropna(subset=["gdp"])
    .sort_values("year")
    .groupby("country")
    .tail(1)
)
# Drop with missing values
df_reg2 = df_latest_gdp.dropna(subset=['population', 'gdp', 'co2'])   
df_reg2.shape

(164, 8)

In [86]:
# Adding more features to the model
# Adding GDP
X = df_reg2[['population', 'gdp']]
y = df_reg2['co2'] 


In [ ]:
#Split Train and Test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)           

In [93]:
# Train multi-factor regression model and generate predictions
model_multi = LinearRegression()
model_multi.fit(X_train, y_train)

# Predictions
pred_multi = model_multi.predict(X_test)



# Evaluate multi-factor model
mae_multi = mean_absolute_error(y_test, pred_multi)
rmse_multi = np.sqrt(mean_squared_error(y_test, pred_multi))
r2_multi = r2_score(y_test, pred_multi)



print('MAE', mae_multi)
print('RMSE', rmse_multi)
print('R-squared', r2_multi)

MAE 99.72223817668846
RMSE 153.9359007414846
R-squared 0.7506354273697191


## Model B2 Interpretation

A multiple linear regression model was developed using both population and GDP as predictors of national CO2 emissions.

Model performance improved substantially compared to the population-only regression, achieving an R² value of 0.751. This indicates that 75.1% of emissions variation across countries can be explained by combined demographic and economic factors.

The reduction in MAE and RMSE further demonstrates improved predictive accuracy.

These findings suggest that while population size contributes to emissions totals, economic output significantly amplifies environmental impact through industrial production, energy consumption, and infrastructure development.

## Model Comparison Evaluation

Two regression models were developed to assess the drivers of national CO2 emissions.

Model B1 (Population Only)

This simple linear regression model used population as the sole predictor of emissions.

Although correlation analysis indicated a strong positive relationship (r = 0.824), the model performed poorly when evaluated using a train-test split. The negative R² value (-0.429) suggests that population alone does not sufficiently explain emissions variation across countries.

This indicates that while population contributes to total emissions, it is not an adequate standalone predictor of emissions responsibility


Model B2  ( Population and GDP)

A multiple linear regression model was then developed incorporating both population and GDP.

This model achieved a significantly improved R² value of 0.751, meaning that 75.1% of emissions variation can be explained by combined demographic and economic factors.

The reduction in MAE and RMSE further confirms improved predictive performance.


## Export to Dashboard 


In [ ]:
## Creating a model comparison table
model_comparison = pd.DataFrame({'Model': ['Population Only', 'Population + GDP'],
                                 'MAE': [mae_pop, mae_multi],
                                        'RMSE': [rmse_pop, rmse_multi],
                                        'R-squared': [r2_pop, r2_multi]})
model_comparison